# Inline NIR Diversion-Control Performance Verification - Simulated

## Purpose

This notebook simulates inline NIR diversion-control performance verification for teaching and study-design exploration. Users edit one configuration cell to define preliminary criteria and simulation assumptions before the notebook generates a historical benchmark, a prospective low/target/high new run, extracted reference samples, and spectral diagnostics.

## Method-specific context

This is not a paired USP <1010> old-versus-new method comparison. The simulated workflow mirrors the inline mother notebook: historical performance is compared with a prospective new run using reference accuracy, diversion-zone checks, spectral diagnostics, and process_repeatability.

In [ ]:
# USER CONFIGURATION - edit values below
METHOD_DISPLAY_NAME = "Inline NIR Diversion Control"
D_EQUIVALENCE_MARGIN = 1.0
K_PRECISION_RATIO = 2.0
ALPHA_ACCURACY = 0.05
ALPHA_PRECISION = 0.05
RECOMMENDED_MIN_N = 18

RANDOM_SEED = 12345
N_SIM = 500

N_TIMEPOINTS = 300
N_REFERENCE_SAMPLES = 24

OLD_MEAN = 100.0
OLD_SD = 0.9
NEW_MEAN = 100.05
NEW_SD = 0.85

LOW_LEVEL = 90.0
TARGET_LEVEL = 100.0
HIGH_LEVEL = 110.0

LOWER_DIVERSION_LIMIT = 80.0
UPPER_DIVERSION_LIMIT = 120.0

PROCESS_TREND_SD = 0.20
REFERENCE_SD = 0.30
DIAGNOSTIC_NOISE_SD = 0.20

GUARD_BAND = 0.50
Q_RESIDUAL_LIMIT = 1.20
HOTELLING_T2_LIMIT = 8.00
PROCESS_REPEATABILITY_WINDOW_POINTS = 31
PROCESS_REPEATABILITY_MIN_POINTS = 11
PROCESS_REPEATABILITY_WINDOW_SECONDS = None
EXCLUDE_INVALID_SPECTRA = True
EXCLUDE_DIAGNOSTIC_EXCEEDANCES = False
EXPORT_REPORT = False


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from nir_cp.inline_diversion import (
    calculate_reference_errors,
    classify_diversion_zone,
    compare_new_to_historical_run_summary,
    inline_verification_decision,
    simulate_inline_diversion_study,
    summarize_accuracy_vs_reference,
    summarize_diversion_decisions,
    summarize_spectral_diagnostics,
)
from nir_cp.notebook_export import export_notebook_pdf
from nir_cp.process_repeatability import calculate_process_repeatability, process_repeatability_summary_frame
from nir_cp.report_equations import display_equation, display_equation_set
from nir_cp.report_tables import display_report_dataframe
from nir_cp.reporting_text import inline_verification_summary_text

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "notebooks").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

METHOD_NAME = METHOD_DISPLAY_NAME
D_INLINE = D_EQUIVALENCE_MARGIN
K_INLINE = K_PRECISION_RATIO


## Simulate data

In [ ]:
simulated = simulate_inline_diversion_study(
    n_timepoints=N_TIMEPOINTS,
    n_reference_samples=N_REFERENCE_SAMPLES,
    old_mean=OLD_MEAN,
    old_sd=OLD_SD,
    new_mean=NEW_MEAN,
    new_sd=NEW_SD,
    low_level=LOW_LEVEL,
    target_level=TARGET_LEVEL,
    high_level=HIGH_LEVEL,
    lower_diversion_limit=LOWER_DIVERSION_LIMIT,
    upper_diversion_limit=UPPER_DIVERSION_LIMIT,
    process_trend_sd=PROCESS_TREND_SD,
    reference_sd=REFERENCE_SD,
    diagnostic_noise_sd=DIAGNOSTIC_NOISE_SD,
    seed=RANDOM_SEED,
)

historical_df = simulated["historical_reference_df"]
new_run_df = simulated["new_run_df"]
new_reference_df = simulated["new_reference_df"]
run_summary_df = simulated["run_summary_df"]

display_report_dataframe(historical_df, max_rows=10);
display_report_dataframe(new_run_df, max_rows=10);
display_report_dataframe(new_reference_df, max_rows=10);
display_report_dataframe(run_summary_df);

## Data checks

In [ ]:
for name, frame in {
    "historical_df": historical_df,
    "new_run_df": new_run_df,
    "new_reference_df": new_reference_df,
    "run_summary_df": run_summary_df,
}.items():
    if frame.empty:
        raise ValueError(f"{name} must not be empty.")

if len(new_run_df) != N_TIMEPOINTS:
    raise ValueError("new_run_df row count does not match N_TIMEPOINTS.")
if len(new_reference_df) != N_REFERENCE_SAMPLES:
    raise ValueError("new_reference_df row count does not match N_REFERENCE_SAMPLES.")

for column in ["nir_prediction", "diversion_lower_limit", "diversion_upper_limit", "q_residual", "hotelling_t2"]:
    new_run_df[column] = pd.to_numeric(new_run_df[column], errors="raise")
for column in ["nir_prediction", "reference_result"]:
    new_reference_df[column] = pd.to_numeric(new_reference_df[column], errors="raise")
    historical_df[column] = pd.to_numeric(historical_df[column], errors="raise")

display_report_dataframe(
    pd.DataFrame([
        {
            "n_timepoints": len(new_run_df),
            "n_reference_samples": len(new_reference_df),
            "segments": ", ".join(new_run_df["segment"].drop_duplicates().astype(str)),
            "low_level": LOW_LEVEL,
            "target_level": TARGET_LEVEL,
            "high_level": HIGH_LEVEL,
        }
    ]),
    transpose_if_one_row=True,
);

# Statistical theory and decision rules

## Study design

Classical paired USP <1010> procedure comparison is not technically feasible for this inline method because the old and new inline methods cannot measure the same dynamic process stream under identical conditions. This simulated workflow therefore uses historical current-method performance, prospective new-method data, extracted reference samples, spectral diagnostics, diversion-zone assessment, and process_repeatability.

The notebook calls tested functions under `src/nir_cp/` and does not implement independent pass/fail logic.

## Accuracy versus reference

For extracted reference samples, the prediction error is:

In [ ]:
_ = display_equation(r"e_i = Y_{NIR,i} - Y_{REF,i}")

The reported agreement metrics include:

In [ ]:
_ = display_equation_set([
    r"bias = \frac{1}{n}\sum_{i=1}^{n}e_i",
    r"MAE = \frac{1}{n}\sum_{i=1}^{n}|e_i|",
    r"RMSEP = \sqrt{\frac{1}{n}\sum_{i=1}^{n}e_i^2}",
])

These metrics assess agreement between inline NIR predictions and extracted reference-sample results. They do not by themselves constitute a paired old-versus-new USP <1010> procedure comparison.

## Diversion-zone assessment

NIR predictions are classified relative to the lower and upper diversion limits:

In [ ]:
_ = display_equation_set([
    r"Y_{NIR,i} < L_{div}",
    r"L_{div} \le Y_{NIR,i} \le U_{div}",
    r"Y_{NIR,i} > U_{div}",
])

A guard band, when used, flags predictions that remain within the diversion limits but are close enough to a limit to need additional review.

## process_repeatability by local linear detrending

The local model is:

In [ ]:
_ = display_equation(r"Y(t) = \beta_0 + \beta_1t + \epsilon")

The residual is:

In [ ]:
_ = display_equation(r"r_i = Y_i - \hat{Y}_i")

The process_repeatability estimate is:

In [ ]:
_ = display_equation(r"s_r = \sqrt{\frac{\sum_{i=1}^{n_r}(r_i-\bar{r})^2}{n_r-1}}")

This metric estimates short-term residual variability after removing slow local process trends. It is not the raw full-run SD. It is not the SD of NIR-reference errors. Exclusions must be predefined, for example invalid spectra or predefined diagnostic exceedances.

## Values used in this notebook

In [ ]:
values_used = {
    "method name": METHOD_NAME,
    "d_inline": D_INLINE,
    "k_inline": K_INLINE,
    "alpha_accuracy": ALPHA_ACCURACY,
    "alpha_precision": ALPHA_PRECISION,
    "recommended_min_n": RECOMMENDED_MIN_N,
    "n_timepoints": N_TIMEPOINTS,
    "n_reference_samples": N_REFERENCE_SAMPLES,
    "old_mean_assumption": OLD_MEAN,
    "old_sd_assumption": OLD_SD,
    "new_mean_assumption": NEW_MEAN,
    "new_sd_assumption": NEW_SD,
    "low_level": LOW_LEVEL,
    "target_level": TARGET_LEVEL,
    "high_level": HIGH_LEVEL,
    "lower_diversion_limit": LOWER_DIVERSION_LIMIT,
    "upper_diversion_limit": UPPER_DIVERSION_LIMIT,
    "process_trend_sd": PROCESS_TREND_SD,
    "reference_sd": REFERENCE_SD,
    "diagnostic_noise_sd": DIAGNOSTIC_NOISE_SD,
    "guard band": GUARD_BAND,
    "window_points": PROCESS_REPEATABILITY_WINDOW_POINTS,
    "window_seconds": PROCESS_REPEATABILITY_WINDOW_SECONDS,
    "min_points": PROCESS_REPEATABILITY_MIN_POINTS,
    "random seed": RANDOM_SEED,
    "n_sim": N_SIM,
}

display_report_dataframe(pd.DataFrame([values_used]), transpose_if_one_row=True);

## Inline performance verification decision

In [ ]:
historical_errors = calculate_reference_errors(historical_df)
new_run_errors = calculate_reference_errors(new_reference_df)

historical_accuracy = summarize_accuracy_vs_reference(historical_df)
new_accuracy = summarize_accuracy_vs_reference(new_reference_df)

display_report_dataframe(pd.DataFrame([historical_accuracy], index=["historical_old_method"]), index=True, transpose_if_one_row=True);
display_report_dataframe(pd.DataFrame([new_accuracy], index=["new_method_run"]), index=True, transpose_if_one_row=True);
display_report_dataframe(new_run_errors[["reference_sample_id", "nir_prediction", "reference_result", "error", "absolute_error"]], max_rows=10);

## Diversion-zone assessment

In [ ]:
historical_zones = classify_diversion_zone(historical_df, guard_band=GUARD_BAND)
new_reference_zones = classify_diversion_zone(new_reference_df, guard_band=GUARD_BAND)
new_run_zones = classify_diversion_zone(new_run_df, guard_band=GUARD_BAND)

display_report_dataframe(summarize_diversion_decisions(historical_zones));
display_report_dataframe(summarize_diversion_decisions(new_reference_zones));
display_report_dataframe(new_reference_zones[["reference_sample_id", "nir_prediction", "diversion_lower_limit", "diversion_upper_limit", "diversion_zone"]], max_rows=10);

## Calculated process_repeatability by local linear detrending

In [ ]:
calculated_repeatability = calculate_process_repeatability(
    new_run_df,
    window_seconds=PROCESS_REPEATABILITY_WINDOW_SECONDS,
    window_points=PROCESS_REPEATABILITY_WINDOW_POINTS,
    min_points=PROCESS_REPEATABILITY_MIN_POINTS,
    exclude_invalid=EXCLUDE_INVALID_SPECTRA,
    q_col="q_residual",
    q_limit=Q_RESIDUAL_LIMIT,
    t2_col="hotelling_t2",
    t2_limit=HOTELLING_T2_LIMIT,
    exclude_diagnostics=EXCLUDE_DIAGNOSTIC_EXCEEDANCES,
)

run_summary_df = run_summary_df.copy()
run_summary_df.loc[run_summary_df["method_status"] == "new", "process_repeatability"] = calculated_repeatability["process_repeatability"]

display_report_dataframe(process_repeatability_summary_frame(calculated_repeatability));
display_report_dataframe(calculated_repeatability["residuals"], max_rows=10);

## Precomputed process_repeatability comparison

In [ ]:
repeatability_comparison = compare_new_to_historical_run_summary(run_summary_df)
display_report_dataframe(pd.DataFrame([repeatability_comparison]), transpose_if_one_row=True);

## Spectral diagnostics

In [ ]:
historical_diagnostics = summarize_spectral_diagnostics(
    historical_df,
    q_limit=Q_RESIDUAL_LIMIT,
    t2_limit=HOTELLING_T2_LIMIT,
)
new_diagnostics = summarize_spectral_diagnostics(
    new_run_df,
    q_limit=Q_RESIDUAL_LIMIT,
    t2_limit=HOTELLING_T2_LIMIT,
)

display_report_dataframe(pd.DataFrame([historical_diagnostics], index=["historical_old_method"]), index=True, transpose_if_one_row=True);
display_report_dataframe(pd.DataFrame([new_diagnostics], index=["new_method_run"]), index=True, transpose_if_one_row=True);

## Overall preliminary decision

In [ ]:
decision = inline_verification_decision(
    accuracy_summary=new_accuracy,
    repeatability_comparison=repeatability_comparison,
    d_inline=D_INLINE,
    k_inline=K_INLINE,
)
decision.update(
    {
        "process_repeatability_mode": calculated_repeatability["mode"],
        "n_residuals": calculated_repeatability["summary"]["n_residuals"],
        "process_repeatability_window_points": calculated_repeatability["parameters"]["window_points"],
        "process_repeatability_window_seconds": calculated_repeatability["parameters"]["window_seconds"],
    }
)

display(Markdown(inline_verification_summary_text(decision, method_name=METHOD_NAME)))
display_report_dataframe(pd.DataFrame([decision]), transpose_if_one_row=True);

## Supporting plots

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(pd.to_datetime(new_run_df["timestamp"]), new_run_df["nir_prediction"], label="new run NIR")
ax.scatter(pd.to_datetime(new_reference_df["timestamp"]), new_reference_df["reference_result"], label="reference samples", zorder=3)
ax.axhline(LOWER_DIVERSION_LIMIT, linestyle="--", color="tab:red", label="lower diversion limit")
ax.axhline(UPPER_DIVERSION_LIMIT, linestyle="--", color="tab:red", label="upper diversion limit")
ax.set_ylabel("Assay units")
ax.set_title("Simulated inline NIR run")
ax.legend(loc="best")
fig.autofmt_xdate()
fig.tight_layout()
display(fig)

## Probability-of-success simulation

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
simulated_bias = rng.normal(
    loc=NEW_MEAN - OLD_MEAN,
    scale=NEW_SD / np.sqrt(N_REFERENCE_SAMPLES),
    size=N_SIM,
)
simulated_repeatability_ratio = rng.normal(
    loc=NEW_SD / OLD_SD,
    scale=0.08,
    size=N_SIM,
)
simulated_accuracy_pass = abs(simulated_bias) <= D_INLINE
simulated_repeatability_pass = simulated_repeatability_ratio <= K_INLINE
inline_success_summary = {
    "n_sim": N_SIM,
    "assumed_bias": NEW_MEAN - OLD_MEAN,
    "assumed_repeatability_ratio": NEW_SD / OLD_SD,
    "pass_accuracy_probability": simulated_accuracy_pass.mean(),
    "pass_repeatability_probability": simulated_repeatability_pass.mean(),
    "pass_both_probability": (simulated_accuracy_pass & simulated_repeatability_pass).mean(),
}
display_report_dataframe(pd.DataFrame([inline_success_summary]), transpose_if_one_row=True);

## Export report

In [ ]:
if EXPORT_REPORT:
    notebook_path = PROJECT_ROOT / "notebooks" / "04_inline_diversion_performance_verification_simulated.ipynb"
    pdf_path = PROJECT_ROOT / "reports" / "pdf" / "04_inline_diversion_performance_verification_simulated.pdf"
    exported_pdf = export_notebook_pdf(notebook_path, pdf_path, hide_code=True, keep_html=True)
    display(Markdown(f"Exported PDF report: `{exported_pdf}`"))
else:
    display(Markdown("PDF export skipped because `EXPORT_REPORT` is `False`."))